In [ ]:
from distributed_processing.worker import Worker
from distributed_processing.filesystem_connector import FileSystemConnector

from time import sleep

from dotenv import load_dotenv
from os import getenv

In [ ]:
load_dotenv()
NS_PATH = getenv("NS_PATH")

In [ ]:
import logging
logging.getLogger("distributed_processing").setLevel(logging.DEBUG)

In [ ]:
fs_connector = FileSystemConnector(NS_PATH)
fs_connector.with_watchdog = True
# Clean the namespace in case previous runs left anything behind.
fs_connector.clean_namespace()

server = Worker(fs_connector)


In [ ]:
def info():
    return server.worker_id, server.requests_queues

server.add_requests_queue(server.worker_id, {"info":info})

In [ ]:
def add(x, y):
    return x + y

def mul(x, y):
    return x * y

def div(x, y):
    return x / y

def lista(x, y):
    return [x, y]

def tupla(x, y):
    return (x,y)

def dic(x,y):
    return {"a":x, "b":[x,y]}

func_dict1 = {"add" : add,
              "mul" : mul,
              "div" : div,
              "lista" : lista,
              "tupla" : tupla,
              "dic" :dic,
              "sleep": sleep}

server.add_requests_queue("cola_1", func_dict1)


In [ ]:
def hola(nombre, calificativo="listo"):
    return f"Hola {nombre}, eres muy {calificativo}"

func_dict2 = {"hola": hola}

# The methods available in cola_2 are not published in the registry
server.add_requests_queue("cola_2", func_dict2, register=False)

In [ ]:
server.add_python_eval()

"""
# Equivalent to:
import dill
import base64

def eval_py_function(str_fn, args=[], kwargs={}):
    "str_fn encoded in base64 ascii"
    return dill.loads(base64.b64decode(str_fn))(*args, **kwargs)

server.add_requests_queue("py_eval", {"eval_py_function": eval_py_function})
"""



In [ ]:
server.queues_to_register

In [ ]:
# Publish every method of every queue in the registry 
server.update_methods_registry()

In [ ]:
# run() is the choice for developing and testing in a notebook: any connector
# error (network drive not responding, lock timeout, corrupt message...) stops
# the cell and shows the traceback. run(timeout=60) serves requests for 60 s
# and then returns.
#
# For a worker deployed as a service use run_forever(): the error is logged,
# the loop waits with exponential backoff and keeps listening. It stops with
# server.stop() (from another thread or from a registered function).
server.run()
# server.run_forever(backoff=(1, 60))